# 05 - Train Random Forest (RF)

Train Random Forest classifier for toxicity prediction.

**Key Features:**
- Ensemble of decision trees
- Feature importance extraction
- Handles imbalanced data with class_weight

In [1]:
# ============================================================================
# IMPORTS AND CONFIG
# ============================================================================
import os, sys, pickle
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score, accuracy_score
import warnings
warnings.filterwarnings('ignore')

PARAM_GRID = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'class_weight': ['balanced', None]
}

TOXICITY_ENDPOINTS = ['NR-AhR', 'NR-AR', 'NR-AR-LBD', 'NR-Aromatase',
                      'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma',
                      'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53']
MODELS_DIR = '../models/baseline_models'
print("✓ Setup complete")

✓ Setup complete


In [ ]:
# ============================================================================
# TRAINING FUNCTION
# ============================================================================
def train_rf(toxicity_name):
    """Train Random Forest for a single toxicity endpoint."""
    print(f"\nTraining RF for {toxicity_name}...")
    
    cache_path = f'../Data/cache/{toxicity_name}/splits.pkl'
    if not os.path.exists(cache_path):
        return None
    
    with open(cache_path, 'rb') as f:
        data = pickle.load(f)
    
    X_train_val = np.vstack([data['train']['X'], data['val']['X']])
    y_train_val = np.concatenate([data['train']['y'], data['val']['y']])
    X_test, y_test = data['test']['X'], data['test']['y']
    
    # Grid search
    rf = RandomForestClassifier(random_state=42, n_jobs=-1)
    grid_search = GridSearchCV(rf, PARAM_GRID, cv=5, scoring='roc_auc', n_jobs=-1)
    grid_search.fit(X_train_val, y_train_val)
    
    # Evaluate
    best_model = grid_search.best_estimator_
    y_proba = best_model.predict_proba(X_test)[:, 1]
    test_auc = roc_auc_score(y_test, y_proba)
    
    print(f"  n_estimators={best_model.n_estimators}, AUC: {test_auc:.4f}")
    
    # Feature importance
    importance = best_model.feature_importances_
    top_features = np.argsort(importance)[-5:][::-1]
    print(f"  Top features: {top_features}")
    
    # Save
    os.makedirs(f'{MODELS_DIR}/{toxicity_name}', exist_ok=True)
    with open(f'{MODELS_DIR}/{toxicity_name}/RF_model.pkl', 'wb') as f:
        pickle.dump({'model': best_model, 'importance': importance}, f)
    
    return {'test_auc': test_auc}

result = train_rf('NR-AhR')